<a href="https://colab.research.google.com/github/HibaBenHsouna1777/Syst-mes-de-recommandation/blob/main/Sys%20de%20recommandation%20etudiant/encadrant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1 - Install deps (Colab)
!pip -q install lightgbm scikit-learn pandas numpy


Systeme de recommandation des encadrants Ce projet met en place un systeme de recommandation intelligent permettant d’associer chaque etudiant a des encadrants potentiels selon la coherence entre son sujet de PFE, sa filiere, ses competences et le profil des enseignants.

L’objectif principal est de produire, pour chaque etudiant, une liste de recommandations pertinentes et ordonnees, en tenant compte a la fois de la structure academique et de la proximite metier entre les profils.

Principe general Le systeme repose sur une approche hybride combinant :

des informations structurelles issues de la base de donnees des representations textuelles enrichies des etudiants et des encadrants des mesures de similarite semantique des regles de compatibilite metier un score final de classement Donnees utilisees Le notebook exploite plusieurs fichiers CSV exportes depuis la base :

clean_users.csv clean_teachers.csv clean_students_predefined_lists.csv clean_groups.csv clean_sectors.csv clean_departments.csv clean_cv_skills.csv clean_cv_skill_users.csv clean_stages.csv clean_stage_types.csv Ces fichiers permettent de reconstruire les profils et les relations entre etudiants, groupes, secteurs, departements, enseignants et stages.

Identification des profils Encadrants Pour les enseignants, la specialite est recuperee directement depuis la table des enseignants, notamment via :

la specialite metier le departement de rattachement les competences

Etudiants Pour les etudiants, la filiere est reconstruite a partir de la chaine suivante :

groupe -> secteur -> departement Le profil etudiant est ensuite enrichi par :

le sujet de PFE les competences les informations descriptives disponibles

Construction de la recommandation Le systeme compare chaque etudiant aux encadrants compatibles, en utilisant plusieurs signaux :

compatibilite du departement proximite entre competences similarite semantique entre le sujet et le profil de l’enseignant proximite avec la specialite de l’enseignant experience de l’enseignant sur des sujets similaires disponibilite de l’enseignant

emb_sim : similarite semantique entre profil etudiant et profil enseignant skill_jaccard : chevauchement entre les competences specialite_sim : proximite entre le sujet et la specialite de l’encadrant historique_sim : proximite avec les anciens sujets encadres availability : disponibilite de l’enseignant Sortie du systeme Pour chaque etudiant, le notebook genere :

un classement des meilleurs encadrants un top 5 final un score de recommandation des raisons explicatives associees a chaque proposition

In [ ]:
# CELL 2 - Imports
import pandas as pd
import numpy as np
import json
import re
from collections import defaultdict

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics.pairwise import cosine_similarity
import lightgbm as lgb


In [ ]:
import pandas as pd
from pathlib import Path

def repair_embedded_json_csv(src_path, dst_path):
    text = Path(src_path).read_text(encoding='utf-8', errors='replace')
    out = []
    i = 0
    n = len(text)

    while i < n:
        if text.startswith('"{', i):
            j = text.find('}"', i + 2)
            if j != -1:
                field = text[i:j+2]
                inner = field[1:-1].replace('"', '""')
                out.append(f'"{inner}"')
                i = j + 2
                continue

        if text.startswith('"[{', i):
            j = text.find('}]"', i + 3)
            if j != -1:
                field = text[i:j+3]
                inner = field[1:-1].replace('"', '""')
                out.append(f'"{inner}"')
                i = j + 3
                continue

        out.append(text[i])
        i += 1

    Path(dst_path).write_text(''.join(out), encoding='utf-8')

repair_embedded_json_csv('/content/clean_users.csv', '/content/clean_users_fixed.csv')
repair_embedded_json_csv('/content/clean_stages.csv', '/content/clean_stages_fixed.csv')

users = pd.read_csv('/content/clean_users_fixed.csv')
cv_skills = pd.read_csv('/content/clean_cv_skills.csv')
cv_skill_users = pd.read_csv('/content/clean_cv_skill_users.csv')
departments = pd.read_csv('/content/clean_departments.csv')
teachers = pd.read_csv('/content/clean_teachers.csv')
groups = pd.read_csv('/content/clean_groups.csv')
sectors = pd.read_csv('/content/clean_sectors.csv')
students_predefined_lists = pd.read_csv('/content/clean_students_predefined_lists.csv')
stage_types = pd.read_csv('/content/clean_stage_types.csv')
stages = pd.read_csv('/content/clean_stages_fixed.csv')

for n, df in {
    "users": users,
    "cv_skills": cv_skills,
    "cv_skill_users": cv_skill_users,
    "departments": departments,
    "teachers": teachers,
    "groups": groups,
    "sectors": sectors,
    "students_predefined_lists": students_predefined_lists,
    "stage_types": stage_types,
    "stages": stages,
}.items():
    print(n, df.shape)


users (413, 38)
cv_skills (137, 10)
cv_skill_users (1000, 8)
departments (7, 17)
teachers (103, 27)
groups (21, 11)
sectors (610, 20)
students_predefined_lists (314, 58)
stage_types (7, 38)
stages (311, 69)


In [ ]:
def norm_cols(df):
    df.columns = [str(c).strip().lower().replace('\ufeff', '') for c in df.columns]
    return df

for df in [users, cv_skills, cv_skill_users, teachers, students_predefined_lists, groups, sectors, stage_types, stages]:
    norm_cols(df)

def norm_txt(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = unidecode(x)
    x = re.sub(r'[^a-z0-9\s]', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return x

def norm_cin(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = re.sub(r"[^0-9]", "", s)
    return s.zfill(8) if s else ""

def pick_col(df, candidates):
    cols = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in cols:
            return cols[c.lower()]
    return None

def jaccard_list(a, b):
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


In [ ]:
u_id = pick_col(users, ['id'])
u_cin = pick_col(users, ['cin'])
u_email = pick_col(users, ['email'])
u_name = pick_col(users, ['name'])
u_cv_profile = pick_col(users, ['cv_profile'])
u_desc_fr = pick_col(users, ['description_fr'])
u_desc_en = pick_col(users, ['description_en'])

t_id = pick_col(teachers, ['id'])
t_name = pick_col(teachers, ['name'])
t_cin = pick_col(teachers, ['cin'])
t_spec = pick_col(teachers, ['specialite_fr', 'specialite'])
t_cap = pick_col(teachers, ['max_supervising_nb', 'capacity'])
t_dep = pick_col(teachers, ['department_id'])

st_id = pick_col(stages, ['id'])
st_type = pick_col(stages, ['stage_type_id'])
st_cand = pick_col(stages, ['candidat_1_id'])
st_sujet = pick_col(stages, ['sujet', 'subject'])
st_enc = pick_col(stages, ['encadrant_cin'])
st_dep = pick_col(stages, ['department_id'])

sty_id = pick_col(stage_types, ['id'])
sty_fr = pick_col(stage_types, ['designation_fr', 'name'])

sk_id = pick_col(cv_skills, ['id'])
sk_txt = pick_col(cv_skills, ['designation_fr', 'designation_en', 'designation'])

su_user = pick_col(cv_skill_users, ['user_id'])
su_skill = pick_col(cv_skill_users, ['cv_skill_id'])

spl_cin = pick_col(students_predefined_lists, ['cin'])
spl_email = pick_col(students_predefined_lists, ['email'])
spl_group = pick_col(students_predefined_lists, ['groupe_id'])

g_id = pick_col(groups, ['id'])
g_sector = pick_col(groups, ['sector_id'])

s_id = pick_col(sectors, ['id'])
s_dep = pick_col(sectors, ['department_id'])

print({
    "users": [u_id, u_cin, u_email, u_name, u_cv_profile, u_desc_fr, u_desc_en],
    "teachers": [t_id, t_name, t_cin, t_spec, t_cap, t_dep],
    "stages": [st_id, st_type, st_cand, st_sujet, st_enc, st_dep]
})


{'users': ['id', 'cin', 'email', 'name', 'cv_profile', 'description_fr', 'description_en'], 'teachers': ['id', 'name', 'cin', 'specialite_fr', 'max_supervising_nb', 'department_id'], 'stages': ['id', 'stage_type_id', 'candidat_1_id', 'sujet', 'encadrant_cin', 'department_id']}


In [ ]:
users['cin_norm'] = users[u_cin].apply(norm_cin)
teachers['cin_norm'] = teachers[t_cin].apply(norm_cin)

teachers[t_cap] = pd.to_numeric(teachers[t_cap], errors='coerce').fillna(0)
teachers[t_dep] = pd.to_numeric(teachers[t_dep], errors='coerce')
teachers[t_id] = pd.to_numeric(teachers[t_id], errors='coerce')

teachers_ok = teachers[
    teachers[t_id].notna() &
    teachers[t_dep].notna() &
    (teachers[t_cap] > 0)
].copy()

print("teachers_ok:", teachers_ok.shape)


teachers_ok: (50, 28)


In [ ]:
stage_types['_fr'] = stage_types[sty_fr].astype(str).str.lower()
pfe_ids = stage_types.loc[
    stage_types['_fr'].str.contains('pfe', na=False),
    sty_id
].dropna().tolist()

stages_pfe = stages[stages[st_type].isin(pfe_ids)].copy()
stages_pfe = stages_pfe[stages_pfe[st_cand].notna()].copy()
stages_pfe = stages_pfe.rename(columns={st_cand: 'student_id'})

if st_dep in stages_pfe.columns:
    stages_pfe = stages_pfe.rename(columns={st_dep: 'department_id'})
else:
    stages_pfe['department_id'] = np.nan

print("stages_pfe:", stages_pfe.shape)


stages_pfe: (104, 69)


In [ ]:
users_min = users[[u_id, u_cin, u_email]].copy()
spl_min = students_predefined_lists[[spl_cin, spl_email, spl_group]].copy()

m_cin = users_min.merge(
    spl_min[[spl_cin, spl_group]],
    left_on=u_cin,
    right_on=spl_cin,
    how='left'
)

m_mail = users_min.merge(
    spl_min[[spl_email, spl_group]],
    left_on=u_email,
    right_on=spl_email,
    how='left'
)

stu_grp = pd.concat([
    m_cin[[u_id, spl_group]].rename(columns={u_id: 'student_id', spl_group: 'groupe_id'}),
    m_mail[[u_id, spl_group]].rename(columns={u_id: 'student_id', spl_group: 'groupe_id'})
], ignore_index=True).dropna().drop_duplicates()

g_map = groups[[g_id, g_sector]].rename(columns={g_id: 'groupe_id', g_sector: 'sector_id'})
s_map = sectors[[s_id, s_dep]].rename(columns={s_id: 'sector_id', s_dep: 'department_id_from_spl'})

stu_grp['groupe_id'] = pd.to_numeric(stu_grp['groupe_id'], errors='coerce')
g_map['groupe_id'] = pd.to_numeric(g_map['groupe_id'], errors='coerce')
g_map['sector_id'] = pd.to_numeric(g_map['sector_id'], errors='coerce')
s_map['sector_id'] = pd.to_numeric(s_map['sector_id'], errors='coerce')
s_map['department_id_from_spl'] = pd.to_numeric(s_map['department_id_from_spl'], errors='coerce')

stu_grp = stu_grp[stu_grp['groupe_id'].notna()].copy()
g_map = g_map[g_map['groupe_id'].notna()].copy()
g_map = g_map[g_map['sector_id'].notna()].copy()
s_map = s_map[s_map['sector_id'].notna()].copy()

stu_grp['groupe_id'] = stu_grp['groupe_id'].astype(int)
g_map['groupe_id'] = g_map['groupe_id'].astype(int)
g_map['sector_id'] = g_map['sector_id'].astype(int)
s_map['sector_id'] = s_map['sector_id'].astype(int)

stu_dept = stu_grp.merge(g_map, on='groupe_id', how='left').merge(s_map, on='sector_id', how='left')
stu_dept = stu_dept[['student_id', 'department_id_from_spl']].drop_duplicates()

stages_pfe = stages_pfe.merge(stu_dept, on='student_id', how='left')
stages_pfe['department_id'] = stages_pfe['department_id'].fillna(stages_pfe['department_id_from_spl'])
stages_pfe = stages_pfe.drop(columns=['department_id_from_spl'])
stages_pfe['department_id'] = pd.to_numeric(stages_pfe['department_id'], errors='coerce')
stages_pfe = stages_pfe[stages_pfe['department_id'].notna()].copy()

print("stages_pfe after backfill:", stages_pfe.shape)


stages_pfe after backfill: (104, 69)


In [ ]:
!pip -q install unidecode
from unidecode import unidecode


In [ ]:
from collections import defaultdict

skill_ref = cv_skills[[sk_id, sk_txt]].copy()
skill_ref['skill_txt'] = skill_ref[sk_txt].apply(norm_txt)

su = cv_skill_users[[su_user, su_skill]].copy()

su[su_user] = pd.to_numeric(su[su_user], errors='coerce')
su[su_skill] = pd.to_numeric(su[su_skill], errors='coerce')
skill_ref[sk_id] = pd.to_numeric(skill_ref[sk_id], errors='coerce')

su = su.dropna(subset=[su_user, su_skill])

su = su.merge(
    skill_ref[[sk_id, 'skill_txt']],
    left_on=su_skill,
    right_on=sk_id,
    how='left'
)

su = su.dropna(subset=['skill_txt', su_user])

su[su_user] = su[su_user].astype(int)

user_skills = defaultdict(list)
for _, r in su.iterrows():
    user_skills[r[su_user]].append(r['skill_txt'])

for k in list(user_skills.keys()):
    user_skills[k] = sorted(set(user_skills[k]))

print("users with skills:", len(user_skills))
print(list(user_skills.items())[:5])


users with skills: 202
[(141, ['laravel']), (142, ['laravel']), (590, ['git', 'javascript', 'laravel', 'sql', 'typescript']), (591, ['data analysis', 'project management', 'sql', 'statistiques', 'uml']), (592, ['data analysis', 'linux', 'matlab', 'project management', 'statistiques'])]


In [ ]:
list(user_skills.items())[:3]


[(141, ['laravel']),
 (142, ['laravel']),
 (643, ['machine learning', 'nosql', 'power bi', 'python', 'sql'])]

In [ ]:
stages['enc_cin_norm'] = stages[st_enc].apply(norm_cin)
load_map = stages[stages['enc_cin_norm'] != ""].groupby('enc_cin_norm').size().to_dict()

teacher_by_cin = teachers_ok[[t_id, 'cin_norm']].dropna().copy()
teacher_by_cin[t_id] = teacher_by_cin[t_id].astype(int)
cin_to_tid = dict(zip(teacher_by_cin['cin_norm'], teacher_by_cin[t_id]))

teacher_past_topics = defaultdict(list)
tmp_hist = stages[[st_sujet, st_enc]].copy()
tmp_hist['enc_cin_norm'] = tmp_hist[st_enc].apply(norm_cin)
tmp_hist['topic_norm'] = tmp_hist[st_sujet].apply(norm_txt)
tmp_hist = tmp_hist[(tmp_hist['enc_cin_norm'] != "") & (tmp_hist['topic_norm'] != "")]

for _, r in tmp_hist.iterrows():
    tid = cin_to_tid.get(r['enc_cin_norm'])
    if tid is not None:
        teacher_past_topics[int(tid)].append(r['topic_norm'])

for tid in list(teacher_past_topics.keys()):
    teacher_past_topics[tid] = teacher_past_topics[tid][:20]

print("load_map:", len(load_map))
print("teacher_past_topics:", len(teacher_past_topics))


load_map: 2
teacher_past_topics: 0


In [ ]:
user_profile_cols = [c for c in [u_id, u_name, u_cv_profile, u_desc_fr, u_desc_en] if c is not None]
users_profile = users[user_profile_cols].copy()

if u_cv_profile is None:
    users_profile['cv_profile'] = ''
    u_cv_profile = 'cv_profile'
if u_desc_fr is None:
    users_profile['description_fr'] = ''
    u_desc_fr = 'description_fr'
if u_desc_en is None:
    users_profile['description_en'] = ''
    u_desc_en = 'description_en'
if u_name is None:
    users_profile['name'] = ''
    u_name = 'name'

users_profile = users_profile.rename(columns={u_id: 'student_id'})


In [ ]:
stu_base = stages_pfe[['student_id', 'department_id', st_sujet]].copy()
stu_base = stu_base.merge(users_profile, on='student_id', how='left')

for col in [u_cv_profile, u_desc_fr, u_desc_en]:
    if col not in stu_base.columns:
        stu_base[col] = ''

stu_base['student_skills'] = stu_base['student_id'].apply(
    lambda x: user_skills.get(int(x), []) if pd.notna(x) else []
)

stu_base['student_text'] = (
    stu_base[st_sujet].fillna('').apply(norm_txt) + ' ' +
    stu_base[u_cv_profile].fillna('').apply(norm_txt) + ' ' +
    stu_base[u_desc_fr].fillna('').apply(norm_txt) + ' ' +
    stu_base[u_desc_en].fillna('').apply(norm_txt) + ' ' +
    stu_base['student_skills'].apply(lambda x: ' '.join(x))
).str.strip()

print("stu_base:", stu_base.shape)
print(stu_base[['student_id', st_sujet, 'student_skills', 'student_text']].head(3).to_string(index=False))


stu_base: (104, 9)
 student_id                                                                                      sujet                                                  student_skills                                                                                                                            student_text
         91                   Etude et interpretation de donnees biologiques pour l aide au diagnostic [data analysis, machine learning, pandas, python, statistiques]   etude et interpretation de donnees biologiques pour l aide au diagnostic    data analysis machine learning pandas python statistiques
         95       Analyse comparative de techniques de laboratoire appliquees a un probleme biologique              [git, machine learning, python, sql, statistiques]    analyse comparative de techniques de laboratoire appliquees a un probleme biologique    git machine learning python sql statistiques
        107 Mise en place d une methodologie de suivi microbiologique da

In [ ]:
print("t columns:", list(t.columns))


t columns: ['id', 'cin', 'nom', 'nom_ar', 'prenom', 'prenom_ar', 'name', 'specialite_fr', 'specialite_en', 'specialite_ar', 'department_id', 'statut_fr', 'statut_en', 'statut_ar', 'created_by', 'updated_by', 'created_at', 'updated_at', 'poste', 'identifiant_unique', 'affichage_in_front', 'grade_id', 'nb_surveillance', 'grade_name', 'max_supervising_nb', 'limit_access_date', 'cin_norm', 'teacher_user_id', '_skills_count', 'availability_fallback']


In [ ]:
print("stages_pfe cols:", stages_pfe.columns.tolist())
print("users_profile cols:", users_profile.columns.tolist())
print("u_cv_profile:", u_cv_profile)
print("u_desc_fr:", u_desc_fr)
print("u_desc_en:", u_desc_en)


stages_pfe cols: ['id', 'student_id', 'candidat_2_id', 'candidat_3_id', 'entreprise_id', 'candidat_1_name', 'candidat_2_name', 'candidat_2_accept', 'candidat_3_name', 'candidat_3_accept', 'encadrant_name', 'sujet', 'domaine', 'sector_id', 'description', 'duree', 'start', 'end', 'rapport', 'level', 'etat', 'created_by', 'updated_by', 'created_at', 'updated_at', 'encadrant_cin', 'avis_encadrant', 'avis_jury', 'remarque_jury', 'resultat_jury', 'remarque_encadrant', 'slug', 'responsible_id', 'status', 'type_stage', 'motivation_letter', 'offer_id', 'company_acceptation', 'teacher_acceptation', 'acceptation', 'company_validation', 'teacher_validation', 'validation', 'assignment_file', 'assignment_file_en', 'internship_certificate', 'activities_file', 'stage_type_id', 'type_bourse', 'deleted_at', 'other_file', 'company_request_sheet', 'company_request_sheet_en', 'appui_request', 'appui_request_en', 'responsible_name', 'responsible_email', 'year_id', 'convention_file', 'journal_stage_file', 'c

In [ ]:
teachers_ok['teacher_skills'] = teachers_ok[t_id].apply(
    lambda x: user_skills.get(int(x), []) if pd.notna(x) else []
)

teachers_ok['past_topics_text'] = teachers_ok[t_id].apply(
    lambda x: ' '.join(teacher_past_topics.get(int(x), []))
)

teachers_ok['teacher_text'] = (
    teachers_ok[t_spec].fillna('').apply(norm_txt) + ' ' +
    teachers_ok['teacher_skills'].apply(lambda x: ' '.join(x)) + ' ' +
    teachers_ok['past_topics_text'].fillna('').apply(norm_txt)
).str.strip()

teachers_ok['skills_count'] = teachers_ok['teacher_skills'].apply(len)
teachers_ok['past_topics_count'] = teachers_ok['past_topics_text'].apply(lambda x: len(str(x).split()) if str(x).strip() else 0)
teachers_ok['teacher_text_len'] = teachers_ok['teacher_text'].apply(lambda x: len(str(x).split()) if pd.notna(x) else 0)

print("teachers with skills:", (teachers_ok['skills_count'] > 0).sum())
print("teachers with past topics:", (teachers_ok['past_topics_count'] > 0).sum())
print(teachers_ok[[t_id, t_name, t_spec, 'teacher_skills', 'past_topics_text']].head(10).to_string(index=False))


teachers with skills: 50
teachers with past topics: 0
 id            name            specialite_fr                                                                                          teacher_skills past_topics_text
540  Firas Bouazizi         Genie Electrique       [automatique, electronique de puissance, electrotechnique, instrumentation, systemes electriques]                 
541    Rania Gharbi           Genie Chimique [cinetique chimique, operations unitaires, procedes industriels, thermodynamique, transfert de chaleur]                 
542    Meriem Zaidi           Biotechnologie            [biochimie, biologie moleculaire, biostatistiques, microbiologie, techniques de laboratoire]                 
543     Sarra Mekki Mathematiques Appliquees                         [algebre lineaire, analyse numerique, optimisation, probabilites, statistiques]                 
544     Nour Jaziri        Marketing Digital                                [branding, content marketing, google ana

In [ ]:
!pip -q install sentence-transformers
from sentence_transformers import SentenceTransformer


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
rows = []

for _, srow in stu_base.iterrows():
    sid = int(srow['student_id'])
    sdept = int(srow['department_id'])
    sujet = norm_txt(srow[st_sujet])
    sskills = srow['student_skills']
    stext = srow['student_text']

    same_dept_teachers = teachers_ok[pd.to_numeric(teachers_ok[t_dep], errors='coerce') == sdept]

    for _, tr in same_dept_teachers.iterrows():
        tid = int(tr[t_id])
        tcin = norm_cin(tr[t_cin])
        cap = float(tr[t_cap])
        load = float(load_map.get(tcin, 0))
        availability = max(0.0, 1.0 - load / max(cap, 1.0))

        rows.append({
            'student_id': sid,
            'teacher_id': tid,
            'teacher_name': tr[t_name],
            'teacher_specialite': tr[t_spec],
            'sujet': sujet,
            'student_skills': sskills,
            'teacher_skills': tr['teacher_skills'],
            'student_text': stext,
            'teacher_text': tr['teacher_text'],
            'teacher_past_topics': tr['past_topics_text'],
            'availability': availability
        })

pairs = pd.DataFrame(rows)
print("pairs:", pairs.shape)
print(pairs.head())


pairs: (1161, 11)
   student_id  teacher_id      teacher_name teacher_specialite  \
0          91         541      Rania Gharbi     Genie Chimique   
1          91         542      Meriem Zaidi     Biotechnologie   
2          91         553  Walid Khalfallah     Genie Chimique   
3          91         554     Omar Ben Amor     Biotechnologie   
4          91         565     Asma Trabelsi     Genie Chimique   

                                               sujet  \
0  etude et interpretation de donnees biologiques...   
1  etude et interpretation de donnees biologiques...   
2  etude et interpretation de donnees biologiques...   
3  etude et interpretation de donnees biologiques...   
4  etude et interpretation de donnees biologiques...   

                                      student_skills  \
0  [data analysis, machine learning, pandas, pyth...   
1  [data analysis, machine learning, pandas, pyth...   
2  [data analysis, machine learning, pandas, pyth...   
3  [data analysis, machi

In [ ]:
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

student_unique = pairs[['student_id', 'student_text']].drop_duplicates().reset_index(drop=True)
teacher_unique = pairs[['teacher_id', 'teacher_text', 'teacher_specialite', 'teacher_past_topics']].drop_duplicates().reset_index(drop=True)

print("student_unique:", student_unique.shape)
print("teacher_unique:", teacher_unique.shape)

student_emb = model.encode(
    student_unique['student_text'].fillna('').astype(str).tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

teacher_emb = model.encode(
    teacher_unique['teacher_text'].fillna('').astype(str).tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

spec_emb = model.encode(
    teacher_unique['teacher_specialite'].fillna('').astype(str).apply(norm_txt).tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

hist_emb = model.encode(
    teacher_unique['teacher_past_topics'].fillna('').astype(str).tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

stu_emb_map = dict(zip(student_unique['student_id'], student_emb))
tea_emb_map = dict(zip(teacher_unique['teacher_id'], teacher_emb))
spec_emb_map = dict(zip(teacher_unique['teacher_id'], spec_emb))
hist_emb_map = dict(zip(teacher_unique['teacher_id'], hist_emb))

print("embedding maps ready")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


student_unique: (104, 2)
teacher_unique: (42, 4)


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

embedding maps ready


In [ ]:
emb_sim = []
skill_jacc = []
specialite_sim = []
historique_sim = []

for _, r in pairs.iterrows():
    sid = r['student_id']
    tid = r['teacher_id']

    svec = stu_emb_map[sid].reshape(1, -1)
    tvec = tea_emb_map[tid].reshape(1, -1)
    spvec = spec_emb_map[tid].reshape(1, -1)
    hvec = hist_emb_map[tid].reshape(1, -1)

    emb_sim.append(float(cosine_similarity(svec, tvec)[0, 0]))
    specialite_sim.append(float(cosine_similarity(svec, spvec)[0, 0]))
    historique_sim.append(float(cosine_similarity(svec, hvec)[0, 0]) if str(r['teacher_past_topics']).strip() else 0.0)
    skill_jacc.append(jaccard_list(r['student_skills'], r['teacher_skills']))

pairs['emb_sim'] = emb_sim
pairs['skill_jaccard'] = skill_jacc
pairs['specialite_sim'] = specialite_sim
pairs['historique_sim'] = historique_sim

print(pairs[['emb_sim', 'skill_jaccard', 'specialite_sim', 'historique_sim', 'availability']].describe())


           emb_sim  skill_jaccard  specialite_sim  historique_sim  \
count  1161.000000     1161.00000     1161.000000          1161.0   
mean      0.321172        0.06948        0.341819             0.0   
std       0.203891        0.22522        0.156214             0.0   
min      -0.007648        0.00000        0.046950             0.0   
25%       0.195634        0.00000        0.184242             0.0   
50%       0.262422        0.00000        0.350598             0.0   
75%       0.420680        0.00000        0.492394             0.0   
max       0.849360        1.00000        0.613214             0.0   

       availability  
count        1161.0  
mean            1.0  
std             0.0  
min             1.0  
25%             1.0  
50%             1.0  
75%             1.0  
max             1.0  


In [ ]:
pairs['score_raw'] = (
    0.40 * pairs['emb_sim'] +
    0.25 * pairs['skill_jaccard'] +
    0.15 * pairs['specialite_sim'] +
    0.10 * pairs['historique_sim'] +
    0.10 * pairs['availability']
)

mn, mx = pairs['score_raw'].min(), pairs['score_raw'].max()
pairs['score'] = (pairs['score_raw'] - mn) / (mx - mn) if mx > mn else 0.5

print(pairs[['score_raw', 'score']].describe())


         score_raw        score
count  1161.000000  1161.000000
mean      0.297112     0.287349
std       0.145547     0.216554
min       0.103983     0.000000
25%       0.205761     0.151431
50%       0.269678     0.246531
75%       0.321868     0.324182
max       0.776088     1.000000


In [ ]:
def make_reasons(r):
    parts = []

    if r['emb_sim'] >= 0.60:
        parts.append("forte similarite semantique sujet profil")
    elif r['emb_sim'] >= 0.35:
        parts.append("similarite semantique moyenne")
    else:
        parts.append("similarite semantique faible")

    if r['skill_jaccard'] >= 0.40:
        parts.append("bon chevauchement de competences")
    elif r['skill_jaccard'] >= 0.15:
        parts.append("quelques competences communes")

    if r['specialite_sim'] >= 0.40:
        parts.append("specialite proche du sujet")

    if r['historique_sim'] >= 0.30:
        parts.append("experience sur des sujets similaires")

    if r['availability'] >= 0.70:
        parts.append("encadrant tres disponible")
    elif r['availability'] >= 0.40:
        parts.append("encadrant disponible")
    else:
        parts.append("encadrant charge")

    common = sorted(set(r['student_skills']) & set(r['teacher_skills']))
    if common:
        parts.append("skills communs: " + ", ".join(common[:4]))

    return " | ".join(parts)

pairs['reasons'] = pairs.apply(make_reasons, axis=1)


In [ ]:
pairs = pairs.sort_values(
    by=['student_id', 'score', 'emb_sim', 'skill_jaccard', 'historique_sim', 'availability', 'teacher_id'],
    ascending=[True, False, False, False, False, False, True]
)

payload = []
for sid, g in pairs.groupby('student_id'):
    top5 = g.head(5)
    payload.append({
        "student_id": int(sid),
        "recommendations": [
            {
                "teacher_id": int(r['teacher_id']),
                "teacher_name": str(r['teacher_name']) if pd.notna(r['teacher_name']) else None,
                "teacher_specialite": str(r['teacher_specialite']) if pd.notna(r['teacher_specialite']) else None,
                "score": float(round(r['score'], 6)),
                "reasons": str(r['reasons'])
            }
            for _, r in top5.iterrows()
        ]
    })

print("students exported:", len(payload))
payload[:2]


students exported: 104


[{'student_id': 91,
  'recommendations': [{'teacher_id': 578,
    'teacher_name': 'Nabil Kefi',
    'teacher_specialite': 'Biotechnologie',
    'score': 0.285245,
    'reasons': 'similarite semantique moyenne | encadrant tres disponible'},
   {'teacher_id': 554,
    'teacher_name': 'Omar Ben Amor',
    'teacher_specialite': 'Biotechnologie',
    'score': 0.279197,
    'reasons': 'similarite semantique moyenne | encadrant tres disponible'},
   {'teacher_id': 566,
    'teacher_name': 'Ines Chaabane',
    'teacher_specialite': 'Biotechnologie',
    'score': 0.264087,
    'reasons': 'similarite semantique moyenne | encadrant tres disponible'},
   {'teacher_id': 542,
    'teacher_name': 'Meriem Zaidi',
    'teacher_specialite': 'Biotechnologie',
    'score': 0.25848,
    'reasons': 'similarite semantique moyenne | encadrant tres disponible'},
   {'teacher_id': 565,
    'teacher_name': 'Asma Trabelsi',
    'teacher_specialite': 'Genie Chimique',
    'score': 0.02577,
    'reasons': 'similari

In [ ]:
teacher_ids_check = [553, 565, 577, 589, 554]

print(
    teachers_ok[teachers_ok[t_id].isin(teacher_ids_check)][
        [t_id, t_name, t_spec, 'teacher_skills', 'past_topics_text', 'teacher_text']
    ].to_string(index=False)
)


 id             name  specialite_fr                                                                                          teacher_skills past_topics_text                                                                                                     teacher_text
553 Walid Khalfallah Genie Chimique                       [cinetique chimique, operations unitaires, thermodynamique, transfert de chaleur]                                       genie chimique cinetique chimique operations unitaires thermodynamique transfert de chaleur
554    Omar Ben Amor Biotechnologie       [analyse biologique, biochimie, biologie moleculaire, biostatistiques, techniques de laboratoire]                        biotechnologie analyse biologique biochimie biologie moleculaire biostatistiques techniques de laboratoire
565    Asma Trabelsi Genie Chimique                       [cinetique chimique, operations unitaires, procedes industriels, thermodynamique]                                       genie chimiq

In [ ]:
teacher_ids_to_check = [542, 554, 566, 578, 541]

check = audit_teachers[audit_teachers[t_id].isin(teacher_ids_to_check)].copy()
print(check[[t_id, t_name, t_spec, 'skills_count', 'past_topics_count', 'teacher_skills', 'past_topics_text', 'teacher_text']].to_string())


     id           name   specialite_fr  skills_count  past_topics_count teacher_skills past_topics_text    teacher_text
54  541   Rania Gharbi  Genie Chimique             0                  0             []                   genie chimique
55  542   Meriem Zaidi  Biotechnologie             0                  0             []                   biotechnologie
67  554  Omar Ben Amor  Biotechnologie             0                  0             []                   biotechnologie
79  566  Ines Chaabane  Biotechnologie             0                  0             []                   biotechnologie
91  578     Nabil Kefi  Biotechnologie             0                  0             []                   biotechnologie


In [ ]:
sid = 91
debug_cols = [
    'student_id', 'teacher_id', 'teacher_name', 'teacher_specialite',
    'emb_sim', 'skill_jaccard', 'specialite_sim', 'historique_sim',
    'availability', 'score', 'reasons'
]

print(
    pairs[pairs['student_id'] == sid][debug_cols]
    .sort_values('score', ascending=False)
    .head(10)
    .to_string(index=False)
)


 student_id  teacher_id     teacher_name teacher_specialite   emb_sim  skill_jaccard  specialite_sim  historique_sim  availability    score                                                   reasons
         91         578       Nabil Kefi     Biotechnologie  0.400070            0.0        0.237799             0.0           1.0 0.285245 similarite semantique moyenne | encadrant tres disponible
         91         554    Omar Ben Amor     Biotechnologie  0.389908            0.0        0.237799             0.0           1.0 0.279197 similarite semantique moyenne | encadrant tres disponible
         91         566    Ines Chaabane     Biotechnologie  0.364519            0.0        0.237799             0.0           1.0 0.264087 similarite semantique moyenne | encadrant tres disponible
         91         542     Meriem Zaidi     Biotechnologie  0.355098            0.0        0.237799             0.0           1.0 0.258480 similarite semantique moyenne | encadrant tres disponible
         9

In [ ]:
print(
    pairs[pairs['student_id'] == 91][['student_id', 'sujet', 'student_text']]
    .drop_duplicates()
    .to_string(index=False)
)


 student_id                                                                    sujet                                                                                                                          student_text
         91 etude et interpretation de donnees biologiques pour l aide au diagnostic etude et interpretation de donnees biologiques pour l aide au diagnostic    data analysis machine learning pandas python statistiques


In [ ]:
print("teachers cin sample:")
print(teachers[[t_id, t_name, t_cin, 'cin_norm']].head(10).to_string(index=False))

print("users cin sample:")
print(users[[u_id, u_name, u_cin, 'cin_norm']].head(10).to_string(index=False))

teacher_cins = set(teachers['cin_norm'].dropna().astype(str))
user_cins = set(users['cin_norm'].dropna().astype(str))

common_cins = teacher_cins & user_cins
print("teachers cin count:", len(teacher_cins))
print("users cin count:", len(user_cins))
print("common cin count:", len(common_cins))
print("sample common cins:", list(sorted(common_cins))[:20])


teachers cin sample:
 id                 name      cin cin_norm
 11     Ahmed ben Hsouna 12121212 12121212
 19    Foulen Ben Foulen 32132132 32132132
289 Dr Enseignant Module 33334444 33334444
490     Youssef Trabelsi 93000001 93000001
491       Amine Chaabane 93000002 93000002
492       Karim Masmoudi 93000003 93000003
493     Walid Khalfallah 93000004 93000004
494        Omar Ben Amor 93000005 93000005
495        Sami Bouazizi 93000006 93000006
496         Bilel Gharbi 93000007 93000007
users cin sample:
 id                            name        cin cin_norm
  8                       Personnel 12345678.0 12345678
 92                ben hsouna Ahmed 12121212.0 12121212
 93               Foulen Ben Foulen 32132132.0 32132132
108                   Foulen FOUKEN 14052270.0 14052270
139                   Etudiant Test 14628561.0 14628561
140                      Admin Test        NaN         
141 Administrateur Plateforme Stage 11112222.0 11112222
142            Dr Enseignant Module 3333

In [ ]:
teacher_ids = set(pd.to_numeric(teachers[t_id], errors='coerce').dropna().astype(int))
user_ids = set(pd.to_numeric(users[u_id], errors='coerce').dropna().astype(int))
skill_user_ids = set(pd.to_numeric(cv_skill_users[su_user], errors='coerce').dropna().astype(int))

print("teacher ids overlapping cv_skill_users:", len(skill_user_ids & teacher_ids))
print("user ids overlapping cv_skill_users:", len(skill_user_ids & user_ids))
print("sample overlap teacher ids:", list(sorted(skill_user_ids & teacher_ids))[:20])
print("sample overlap user ids:", list(sorted(skill_user_ids & user_ids))[:20])


teacher ids overlapping cv_skill_users: 48
user ids overlapping cv_skill_users: 408
sample overlap teacher ids: [289, 543, 544, 545, 546, 547, 548, 549, 550, 551, 552, 553, 554, 555, 556, 557, 558, 559, 560, 561]
sample overlap user ids: [108, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159]


In [ ]:
teachers_ok_v2 = teachers_ok.copy()

teachers_ok_v2['teacher_skills_direct'] = teachers_ok_v2[t_id].apply(
    lambda x: user_skills.get(int(x), []) if pd.notna(x) else []
)

teachers_ok_v2['direct_skills_count'] = teachers_ok_v2['teacher_skills_direct'].apply(len)

print(teachers_ok_v2[[t_id, t_name, 'teacher_skills_direct', 'direct_skills_count']].head(20).to_string(index=False))
print("teachers with direct skills:", (teachers_ok_v2['direct_skills_count'] > 0).sum())


 id             name                                            teacher_skills_direct  direct_skills_count
540   Firas Bouazizi                                                               []                    0
541     Rania Gharbi                                                               []                    0
542     Meriem Zaidi                                                               []                    0
543      Sarra Mekki      [data analysis, project management, sql, statistiques, uml]                    5
544      Nour Jaziri [data analysis, linux, matlab, project management, statistiques]                    5
545      Asma Haddad       [linux, machine learning, project management, python, sql]                    5
546      Ines Brahmi   [data analysis, project management, python, sql, statistiques]                    5
547      Aya Mabrouk     [data analysis, machine learning, python, sql, statistiques]                    5
548         Rim Kefi     [data analys

In [ ]:
# CELL I - Top5 export JSON (Laravel compatible)
pairs = pairs.sort_values(
    by=['student_id','score','skill_jaccard','emb_sim','availability','teacher_id'],
    ascending=[True,False,False,False,False,True]
)

payload = []
for sid, g in pairs.groupby('student_id'):
    top5 = g.head(5)
    recs = []
    for _, r in top5.iterrows():
        recs.append({
            "teacher_id": int(r['teacher_id']),
            "teacher_name": str(r['teacher_name']) if pd.notna(r['teacher_name']) else None,
            "teacher_specialite": str(r['teacher_specialite']) if pd.notna(r['teacher_specialite']) else None,
            "score": float(round(r['score'], 6)),
            "reasons": str(r['reasons'])
        })
    payload.append({
        "student_id": int(sid),
        "recommendations": recs
    })

out_path = '/content/recommendations_top5_hybrid_prod.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("students exported:", len(payload))
print("file:", out_path)
print(payload[0] if payload else "empty")


students exported: 104
file: /content/recommendations_top5_hybrid_prod.json
{'student_id': 91, 'recommendations': [{'teacher_id': 542, 'teacher_name': 'Meriem Zaidi', 'teacher_specialite': 'Biotechnologie', 'score': 1.0, 'reasons': 'similarite semantique faible | fort chevauchement de competences | encadrant tres disponible'}, {'teacher_id': 554, 'teacher_name': 'Omar Ben Amor', 'teacher_specialite': 'Biotechnologie', 'score': 1.0, 'reasons': 'similarite semantique faible | fort chevauchement de competences | encadrant tres disponible'}, {'teacher_id': 566, 'teacher_name': 'Ines Chaabane', 'teacher_specialite': 'Biotechnologie', 'score': 1.0, 'reasons': 'similarite semantique faible | fort chevauchement de competences | encadrant tres disponible'}, {'teacher_id': 578, 'teacher_name': 'Nabil Kefi', 'teacher_specialite': 'Biotechnologie', 'score': 1.0, 'reasons': 'similarite semantique faible | fort chevauchement de competences | encadrant tres disponible'}, {'teacher_id': 541, 'teacher_